# 05 — KPI Framework & Tableau-Ready Export
**Purpose:** Compute dashboard KPIs + export Tableau-ready CSVs from the cleaned, market-sale-filtered data.

## KPI framework

| KPI | Definition | Why |
|---|---|---|
| Median Sale Price | median of segment | Robust to heavy tail |
| Median $/sqft | median of segment | Size-controlled price comparison |
| Transaction Volume | count of rows | Market liquidity |
| Affordability Index | segment median / citywide median | <1 cheaper than average, >1 pricier |
| Price IQR | Q3 − Q1 on price | Market volatility proxy |
| Median Age | median of building age | Housing stock vintage (median, not mean — heavy tail) |

## Guardrails
- All KPIs computed on `is_market_sale==True` rows only (excludes deed transfers that distort every aggregate).
- Neighborhood KPIs require **≥20 transactions** for statistical stability.
- Age uses **median** (right-skewed + some zero/unknown). Price uses median (skew ≈ 20).


In [1]:
import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path('/Users/krishnadevan/Documents/dvacapstone2')
OUT  = ROOT/'data'/'processed'; OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ROOT/'data'/'nyc_cleaned.parquet')
mk = df[df['is_market_sale']].dropna(subset=['SALE PRICE']).copy()

# Consistent column names for downstream dashboard use
mk = mk.rename(columns={
    'BOROUGH_NAME':'borough_name',
    'NEIGHBORHOOD':'neighborhood',
    'BUILDING CLASS CATEGORY':'building_class',
    'TAX CLASS AT PRESENT':'tax_class',
    'SALE PRICE':'sale_price',
    'PRICE_PER_SQFT':'price_per_sqft',
    'GROSS SQUARE FEET':'gross_square_feet',
    'LAND SQUARE FEET':'land_square_feet',
    'AGE_AT_SALE':'building_age',
    'SALE DATE':'sale_date',
    'RESIDENTIAL UNITS':'residential_units',
    'COMMERCIAL UNITS':'commercial_units',
    'TOTAL UNITS':'total_units',
    'YEAR BUILT':'year_built',
    'ZIP CODE':'zip_code',
    'ADDRESS':'address',
})
mk['sale_year']  = mk['sale_date'].dt.year
mk['sale_month'] = mk['sale_date'].dt.month
mk['building_class'] = mk['building_class'].str.strip()

citywide_median = mk['sale_price'].median()
citywide_ppsqft = mk['price_per_sqft'].median()
print(f'Market-sale rows    : {len(mk):,}')
print(f'Citywide median     : ${citywide_median:,.0f}')
print(f'Citywide $/sqft     : ${citywide_ppsqft:,.0f}')

Market-sale rows    : 58,713
Citywide median     : $635,163
Citywide $/sqft     : $349


## Borough-level KPIs

In [2]:
borough_kpi = mk.groupby('borough_name').agg(
    median_sale_price     = ('sale_price',       'median'),
    median_price_per_sqft = ('price_per_sqft',   'median'),
    transaction_volume    = ('sale_price',       'count'),
    price_iqr             = ('sale_price',       lambda x: x.quantile(0.75) - x.quantile(0.25)),
    median_gross_sqft     = ('gross_square_feet','median'),
    median_building_age   = ('building_age',     'median'),
    total_sales_value     = ('sale_price',       'sum'),
).round(2)

borough_kpi['affordability_index']      = (borough_kpi['median_sale_price']/citywide_median).round(3)
borough_kpi['affordability_ppsqft_idx'] = (borough_kpi['median_price_per_sqft']/citywide_ppsqft).round(3)
borough_kpi = borough_kpi.sort_values('median_sale_price', ascending=False)
print(borough_kpi.to_string())
borough_kpi.to_csv(OUT/'kpi_borough.csv')
print('saved kpi_borough.csv')

               median_sale_price  median_price_per_sqft  transaction_volume   price_iqr  median_gross_sqft  median_building_age  total_sales_value  affordability_index  affordability_ppsqft_idx
borough_name                                                                                                                                                                                     
Manhattan              1175000.0                 782.31               14252  1821305.25             7507.5                 62.0       4.807588e+10                1.850                     2.243
Brooklyn                770000.0                 401.12               15369   805000.00             2232.0                 86.0       2.005395e+10                1.212                     1.150
Queens                  500000.0                 381.84               18151   485000.00             1616.0                 68.0       1.360283e+10                0.787                     1.095
Staten Island           468468

## Neighborhood-level KPIs (≥20 transactions)

In [3]:
neigh_kpi = mk.groupby(['borough_name','neighborhood']).agg(
    median_sale_price     = ('sale_price','median'),
    median_price_per_sqft = ('price_per_sqft','median'),
    transaction_volume    = ('sale_price','count'),
    median_gross_sqft     = ('gross_square_feet','median'),
    median_building_age   = ('building_age','median'),
    price_iqr             = ('sale_price', lambda x: x.quantile(0.75)-x.quantile(0.25)),
).round(2).reset_index()

neigh_kpi = neigh_kpi[neigh_kpi['transaction_volume']>=20].copy()
neigh_kpi['affordability_index'] = (neigh_kpi['median_sale_price']/citywide_median).round(3)
neigh_kpi = neigh_kpi.sort_values('median_price_per_sqft', ascending=False)

print(f'Neighborhoods with >=20 txns: {len(neigh_kpi)}')
print('\nTop 10 by $/sqft:')
print(neigh_kpi.head(10).to_string(index=False))
print('\nBest value (lowest $/sqft, >=100 txns):')
val = neigh_kpi[neigh_kpi['transaction_volume']>=100].nsmallest(10,'median_price_per_sqft')
print(val.to_string(index=False))

neigh_kpi.to_csv(OUT/'kpi_neighborhood.csv', index=False)
print('\nsaved kpi_neighborhood.csv')

Neighborhoods with >=20 txns: 239

Top 10 by $/sqft:
borough_name              neighborhood  median_sale_price  median_price_per_sqft  transaction_volume  median_gross_sqft  median_building_age  price_iqr  affordability_index
   Manhattan   UPPER EAST SIDE (59-79)          1100000.0                1850.00                1459             6230.0                 58.0 1675000.00                1.732
   Manhattan    GREENWICH VILLAGE-WEST          1400000.0                1814.15                 477             4381.0                 86.0 4730000.00                2.204
   Manhattan GREENWICH VILLAGE-CENTRAL          1395000.0                1770.45                 469             6004.5                 87.0 1545000.00                2.196
   Manhattan                      SOHO          2830093.5                1755.50                 242             8636.5                 97.0 3696250.00                4.456
    Brooklyn      DOWNTOWN-FULTON MALL          1550000.0                1661.68  

## Building-class KPIs

In [4]:
cls_kpi = mk.groupby('building_class').agg(
    median_sale_price     = ('sale_price','median'),
    median_price_per_sqft = ('price_per_sqft','median'),
    transaction_volume    = ('sale_price','count'),
    median_gross_sqft     = ('gross_square_feet','median'),
    median_building_age   = ('building_age','median'),
).round(2).reset_index()
cls_kpi = cls_kpi[cls_kpi['transaction_volume']>=20].sort_values('transaction_volume', ascending=False)
cls_kpi['affordability_index'] = (cls_kpi['median_sale_price']/citywide_median).round(3)
print(cls_kpi.head(15).to_string(index=False))
cls_kpi.to_csv(OUT/'kpi_building_class.csv', index=False)
print('saved kpi_building_class.csv')

                   building_class  median_sale_price  median_price_per_sqft  transaction_volume  median_gross_sqft  median_building_age  affordability_index
          01 ONE FAMILY DWELLINGS           510000.0                 372.22               12724             1378.0                 77.0                0.803
   10 COOPS - ELEVATOR APARTMENTS           430000.0                   1.87               11524           125605.5                 64.0                0.677
  13 CONDOS - ELEVATOR APARTMENTS          1100000.0                    NaN               10266                NaN                 11.0                1.732
          02 TWO FAMILY DWELLINGS           655000.0                 326.93                9912             2040.0                 87.0                1.031
     09 COOPS - WALKUP APARTMENTS           289000.0                   8.01                2504            23712.0                 80.0                0.455
        03 THREE FAMILY DWELLINGS           800000.0      

## Monthly KPIs (time series)

In [5]:
monthly_kpi = mk.groupby(['sale_year','sale_month']).agg(
    median_sale_price     = ('sale_price','median'),
    median_price_per_sqft = ('price_per_sqft','median'),
    transaction_volume    = ('sale_price','count'),
    total_value           = ('sale_price','sum'),
).round(2).reset_index()
monthly_kpi['period'] = pd.to_datetime(
    monthly_kpi[['sale_year','sale_month']].rename(columns={'sale_year':'year','sale_month':'month'}).assign(day=1))
monthly_kpi = monthly_kpi.sort_values('period')
print(monthly_kpi.to_string(index=False))
monthly_kpi.to_csv(OUT/'kpi_monthly.csv', index=False)
print('saved kpi_monthly.csv')

 sale_year  sale_month  median_sale_price  median_price_per_sqft  transaction_volume  total_value     period
      2016           9           600000.0                 339.69                5368 7.836628e+09 2016-09-01
      2016          10           610475.0                 339.80                4508 6.089734e+09 2016-10-01
      2016          11           600000.0                 336.90                4719 7.069896e+09 2016-11-01
      2016          12           618754.5                 344.49                5316 1.052217e+10 2016-12-01
      2017           1           638000.0                 345.08                4738 7.837169e+09 2017-01-01
      2017           2           619000.0                 347.00                4350 5.487072e+09 2017-02-01
      2017           3           608068.5                 338.87                5186 7.475401e+09 2017-03-01
      2017           4           620000.0                 338.97                4393 5.999065e+09 2017-04-01
      2017         

## Borough × Building-class KPIs (heatmap-ready)

In [6]:
top4 = mk['building_class'].value_counts().head(4).index.tolist()
sub = mk[mk['building_class'].isin(top4)].copy()
sub['bcc_short'] = sub['building_class'].str[:25]

bxc_kpi = sub.groupby(['borough_name','bcc_short']).agg(
    median_sale_price     = ('sale_price','median'),
    median_price_per_sqft = ('price_per_sqft','median'),
    transaction_volume    = ('sale_price','count'),
).round(2).reset_index()

print(bxc_kpi.to_string(index=False))
bxc_kpi.to_csv(OUT/'kpi_borough_class.csv', index=False)
print('saved kpi_borough_class.csv')

 borough_name                 bcc_short  median_sale_price  median_price_per_sqft  transaction_volume
        Bronx   01 ONE FAMILY DWELLINGS           400000.0                 268.94                1027
        Bronx   02 TWO FAMILY DWELLINGS           485000.0                 222.39                1351
        Bronx 10 COOPS - ELEVATOR APART           185000.0                   2.17                 968
        Bronx 13 CONDOS - ELEVATOR APAR           136000.0                    NaN                 301
     Brooklyn   01 ONE FAMILY DWELLINGS           680000.0                 460.99                2292
     Brooklyn   02 TWO FAMILY DWELLINGS           825000.0                 390.62                3530
     Brooklyn 10 COOPS - ELEVATOR APART           335000.0                   1.49                1872
     Brooklyn 13 CONDOS - ELEVATOR APAR           933950.0                    NaN                2396
    Manhattan   01 ONE FAMILY DWELLINGS          6750000.0                1698.84 

## Final Tableau-ready dataset (row-level)

In [7]:
tableau_cols = ['borough','borough_name','neighborhood','building_class','tax_class',
                'zip_code','address',
                'residential_units','commercial_units','total_units',
                'gross_square_feet','land_square_feet','year_built','building_age',
                'sale_price','price_per_sqft','sale_date','sale_year','sale_month']
tableau_cols = [c for c in tableau_cols if c in mk.columns]
mk[tableau_cols].to_csv(OUT/'final_tableau_data.csv', index=False)
print(f'Final tableau dataset shape: {mk[tableau_cols].shape}')
print('saved final_tableau_data.csv')

Final tableau dataset shape: (58713, 18)
saved final_tableau_data.csv


## Headline KPI summary

In [8]:
print('='*60)
print('HEADLINE KPIs — NYC PROPERTY SALES (market sales only)')
print('='*60)
print(f'Total market-sale transactions : {len(mk):,}')
print(f'Deed transfers excluded        : {(~df["is_market_sale"]).sum():,}')
print(f'Citywide median sale price     : ${citywide_median:,.0f}')
print(f'Citywide median $/sqft         : ${citywide_ppsqft:,.0f}')
print(f'Citywide mean sale price       : ${mk["sale_price"].mean():,.0f}  (biased by tail)')
print(f'Date range                     : {mk["sale_date"].min().date()} -> {mk["sale_date"].max().date()}')
print()
print('Borough affordability (1.0 = citywide median)')
print(borough_kpi[['median_sale_price','median_price_per_sqft','transaction_volume',
                   'affordability_index','affordability_ppsqft_idx']].to_string())

HEADLINE KPIs — NYC PROPERTY SALES (market sales only)
Total market-sale transactions : 58,713
Deed transfers excluded        : 25,070
Citywide median sale price     : $635,163
Citywide median $/sqft         : $349
Citywide mean sale price       : $1,518,333  (biased by tail)
Date range                     : 2016-09-01 -> 2017-08-31

Borough affordability (1.0 = citywide median)
               median_sale_price  median_price_per_sqft  transaction_volume  affordability_index  affordability_ppsqft_idx
borough_name                                                                                                              
Manhattan              1175000.0                 782.31               14252                1.850                     2.243
Brooklyn                770000.0                 401.12               15369                1.212                     1.150
Queens                  500000.0                 381.84               18151                0.787                     1.095
Sta

---
## Artifacts written (all in `data/processed/`)
- `kpi_borough.csv`
- `kpi_neighborhood.csv`  (≥20 txns)
- `kpi_building_class.csv`
- `kpi_monthly.csv`
- `kpi_borough_class.csv`
- `final_tableau_data.csv`  (row-level, market sales only)

All ready to drop into Tableau / Power BI. Every aggregate respects `is_market_sale` filter so dashboard numbers match statistical analysis numbers.
